In [12]:
#Task 4: Paper Analysis with LLMs
!pip install pymupdf openai pdfplumber
!pip install --upgrade openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.5/720.5 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.81.0
    Uninstalling openai-1.81.0:
      Successfully uninstalled openai-1.81.0


In [3]:
import requests

# Link del pdf
pdf_url = "https://revistasinvestigacion.unmsm.edu.pe/index.php/iigeo/article/download/20638/16775/70290"
pdf_filename = "Cobertura.pdf"

response = requests.get(pdf_url)
with open(pdf_filename, "wb") as f:
    f.write(response.content)

print(f"✅ PDF guardado como {pdf_filename}")

✅ PDF guardado como Cobertura.pdf


In [5]:
import fitz  # PyMuPDF

# Abrir el PDF
doc = fitz.open(pdf_filename)

# Extraer texto de todas las páginas
texto = ""
for page in doc:
    texto += page.get_text()

print("✅ Texto extraído")
print(texto[:2000])  # Mostrar primeros 1000 caracteres

✅ Texto extraído
© Los autores. Este artículo es publicado por la Revista del Instituto de investigación de la Facultad de minas, metalurgia y ciencias 
geográficas de la Universidad Nacional Mayor de San Marcos. Este es un artículo de acceso abierto, distribuido bajo los términos de la 
licencia Creative Commons Atribución 4.0 Internacional (CC BY 4.0) [https://creativecommons.org/licenses/by/4.0/deed.es] que permite 
el uso, distribución y reproducción en cualquier medio, siempre que la obra original sea debidamente citada de su fuente original. Para 
mas información, por favor póngase en contacto con iigeo@unmsm.edu.pe
Rev. Inst. investig. Fac. minas metal. cienc. geogr. vol 24 n° 47, 2021: 13 - 18	
ISSN-L:1561-0888
Cobertura arbórea y captura de dióxido de carbono en los 
parques urbanos. Caso: Lima Norte
Tree cover and carbone dioxide capture in urban parks case: Northern Lima
Walter Aparicio Arévalo Gómez 1 , Francisco Alejandro Alcántara Boza 2
Recibido: 14/10/2020 - Aprobado: 2

In [16]:
import getpass
import openai

# Insertar tu clave de forma segura
api_key = getpass.getpass("🔐 Ingresa tu OpenAI API key: ")
openai_client = openai.OpenAI(api_key=api_key)

🔐 Ingresa tu OpenAI API key: ··········


In [46]:
# Buscar el título y extraer el bloque de texto que sigue
titulo = "Tabla 1."
start_index = texto.find(titulo)

if start_index != -1:
    # Buscar el índice donde aparece la palabra 'Total' después del título
    end_index = texto.find("Total", start_index)

    # Si se encuentra 'Total', extraer desde el título hasta esa línea
    if end_index != -1:
        tabla_texto = texto[start_index + len(titulo):end_index].strip()
    else:
        # Si no se encuentra 'Total', extraer un bloque fijo como respaldo
        tabla_texto = texto[start_index + len(titulo): start_index + len(titulo) + 2000].strip()

    print(tabla_texto)
else:
    print("No se encontró la tabla.")

# Limpiar líneas y dividir
lineas = tabla_texto.strip().splitlines()

# Filtrar datos válidos: cada parque tiene 5 líneas (nombre, área, %, kg/m², total CO₂)
parques = []
i = 0
while i < len(lineas) - 4:
    nombre = lineas[i].strip()
    # Evitar líneas no deseadas como "Total"
    if nombre.lower() == "total":
        break
    try:
        area = float(lineas[i+1].strip())
        cobertura = float(lineas[i+2].strip())
        co2_m2 = float(lineas[i+3].strip())
        co2_total = float(lineas[i+4].strip())

        parques.append({
            "nombre": nombre,
            "area": area,
            "cobertura": cobertura,
            "co2_m2": co2_m2,
            "co2_total": co2_total
        })

        i += 5  # avanzar al siguiente bloque
    except ValueError:
        # Saltar si alguna línea tiene un valor inválido
        i += 1

# Convertir la tabla de parques a texto estructurado
tabla_texto_llm = "\n".join([
    f"{p['nombre']}: área={p['area']} m², cobertura={p['cobertura']}%, CO₂/m²/año={p['co2_m2']}, CO₂ total={p['co2_total']} kg"
    for p in parques
])

# Instrucción al modelo
prompt = f"""A continuación se muestra una tabla de datos de parques urbanos en Lima Norte.
Cada parque incluye su área, el porcentaje de cobertura arbórea, la cantidad de CO₂ capturado por metro cuadrado al año, y el total de CO₂ capturado anualmente.
Realiza un ranking de los parques según su eficiencia (considera la relación entre área, cobertura y CO₂ capturado)
y explica brevemente por qué los más eficientes destacan.

Datos:
{tabla_texto_llm}
"""

# Llamada al modelo
respuesta = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": "Eres un experto en política ambiental urbana y planificación verde."},
        {"role": "user", "content": prompt}
    ]
)

# Mostrar resultado
print("\n📊 Análisis del modelo:")
print(respuesta.choices[0].message.content)

Porcentaje de cobertura arbórea y kilogramos de CO2 por año 
capturado en parques de Lima Norte
Parque
Area del 
Parque
% Cobertura 
Arbórea
KgCO2/m2-año
kg CO2 
almacenado
CO1
4900
54.00
0.591
72746.68
CO2
3100
12.00
0.136
10623.00
CO3
30300
14.00
0.169
128680.00
CO4
5800
54.00
0.570
83057.78
CO5
4900
28.00
0.305
37581.73
CO6
4000
14.00
0.163
16328.96
CO7
8200
17.00
0.191
39384.82
CO8
15000
12.00
0.147
55432.47
CO9
11900
36.00
0.392
116001.96
IN1
2500
70.00
0.774
48587.95
IN2
4000
20.00
0.230
23115.27
IN3
2000
18.00
0.198
9941.84
IN4
1700
62.00
0.671
28662.64
IN5
2850
29.00
0.320
22903.30
LO1
2800
50.00
0.563
39619.90
LO2
10100
13.00
0.140
35500.61
LO3
10000
39.00
0.450
112950.00
LO4
6000
22.00
0.311
46792.42
LO5
10000
41.00
0.444
111510.00
LO6
24000
9.00
0.173
104200.00
LO7
15100
28.00
0.300
113780.00
LO8
13700
12.00
0.161
55229.62
LO9
5500
28.00
0.316
43671.35
SM1
7000
51.00
0.557
97850.00
SM2
16700
18.00
0.201
126690.00
SM3
4000
47.00
0.530
53225.40
SM4
6000
27.00
0.292
43956.52
SM

In [49]:
prompt = f"""
Tengo una tabla con datos de parques urbanos de Lima Norte. Ya está estructurada y agrupada por distrito.

Aquí están los resultados agregados por distrito (distrito, número de parques, área total, kg de CO2 total capturado por año, cobertura arbórea promedio):

{tabla_texto_llm}

Considera que aquellos resultados con que inicien con CO forman parte del distrito de Comas, IN Independencia, LO Los Olivos y SM San Martin de Porres

Con base en estos datos, responde las siguientes preguntas como un experto en gestión urbana sostenible y cambio climático:

1. ¿Qué distritos presentan mejores resultados en cobertura arbórea y captura total de CO₂?
2. ¿Qué distritos están más rezagados y por qué?
3. ¿Qué tipo de especies arbóreas serían óptimas para los distritos con menor cobertura, considerando eficiencia de captura y adaptación urbana?
4. ¿Qué recomendaciones de política pública urbana y ambiental propondrías para mejorar la captura de carbono y cobertura arbórea en Lima Norte?
"""

response = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "Eres un experto en gestión urbana sostenible y cambio climático."},
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=800,
)

print(response.choices[0].message.content)


1. Los distritos que presentan mejores resultados en cobertura arbórea y captura total de CO₂ son San Martin de Porres (SM) y Los Olivos (LO). Ambos distritos tienen una cobertura arbórea promedio relativamente alta y han logrado capturar una cantidad significativa de CO₂, lo que indica una buena gestión de sus parques urbanos en términos de sostenibilidad y mitigación del cambio climático.

2. Los distritos más rezagados en términos de cobertura arbórea y captura de CO₂ son Comas (CO) e Independencia (IN). Estos distritos presentan coberturas arbóreas promedio más bajas y menor captura total de CO₂ en comparación con los otros distritos. Esto puede deberse a una menor inversión en áreas verdes, falta de mantenimiento de los parques existentes o una planificación urbana menos sostenible.

3. Para los distritos con menor cobertura arbórea, sería óptimo considerar especies arbóreas que sean eficientes en la captura de CO₂ y que se adapten bien al entorno urbano. Algunas especies recomend